# RAG（检索增强生成）
## Retrieval-Augmented Generation

<img src="../images/logo.png" width=150>

RAG通过检索外部知识库来增强LLM的生成能力，解决模型知识过时、幻觉等问题。ChatGPT、Claude等主流产品都采用了RAG架构。

RAG enhances LLM generation by retrieving from external knowledge bases, solving issues like outdated knowledge and hallucinations. ChatGPT, Claude and other major products use RAG architecture.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import defaultdict

class Document:
    """文档类 / Document class"""
    def __init__(self, content, metadata=None):
        self.content = content
        self.metadata = metadata or {}
        self.embedding = None

class VectorStore:
    """
    简化向量数据库 / Simplified vector database
    使用余弦相似度进行检索
    Uses cosine similarity for retrieval
    """
    def __init__(self, embed_dim=768):
        self.embed_dim = embed_dim
        self.documents = []
        self.embeddings = []
        self.embed_model = None  # 嵌入模型 / Embedding model
    
    def set_embed_model(self, model):
        """设置嵌入模型 / Set embedding model"""
        self.embed_model = model
    
    def add_documents(self, documents):
        """添加文档并计算嵌入 / Add documents and compute embeddings"""
        for doc in documents:
            self.documents.append(doc)
            if self.embed_model:
                doc.embedding = self.embed_model.encode(doc.content)
                self.embeddings.append(doc.embedding)
        self.embeddings = torch.stack(self.embeddings) if self.embeddings else torch.zeros(0, self.embed_dim)
    
    def search(self, query, top_k=3):
        """
        检索最相关的文档
        Retrieve most relevant documents
        """
        if not self.embeddings.size(0):
            return []
        
        # 计算查询嵌入 / Compute query embedding
        query_embedding = self.embed_model.encode(query).unsqueeze(0)
        
        # 计算余弦相似度 / Compute cosine similarity
        similarities = F.cosine_similarity(query_embedding, self.embeddings, dim=-1)
        
        # 获取top-k / Get top-k
        top_indices = similarities.topk(top_k).indices.tolist()
        
        results = []
        for idx in top_indices:
            results.append({
                'document': self.documents[idx],
                'score': similarities[idx].item()
            })
        
        return results

# 简单的句子嵌入模型 / Simple sentence embedding model
class SimpleEmbeddingModel(nn.Module):
    """简单的词嵌入平均模型 / Simple word embedding averaging model"""
    def __init__(self, vocab_size=10000, embed_dim=768):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
    
    def encode(self, text, max_len=100):
        """将文本编码为向量 / Encode text to vector"""
        # 简化实现：使用词嵌入平均
        # Simplified: use word embedding average
        tokens = text.lower().split()[:max_len]
        token_ids = [hash(t) % self.embedding.num_embeddings for t in tokens]
        
        if not token_ids:
            return torch.zeros(self.embedding.embedding_dim)
        
        token_tensor = torch.tensor(token_ids).unsqueeze(0)
        embed = self.embedding(token_tensor).mean(dim=1).squeeze()
        return embed / embed.norm()  # 归一化 / Normalize

# 测试 / Test
embed_model = SimpleEmbeddingModel(vocab_size=10000, embed_dim=768)

docs = [
    Document("Python is a high-level programming language", {"source": "python.org"}),
    Document("Machine learning is a subset of artificial intelligence", {"source": "wiki"}),
    Document("PyTorch is a deep learning framework", {"source": "pytorch.org"}),
    Document("Transformers are the foundation of modern NLP", {"source": "paper"}),
    Document("RAG combines retrieval with generation for better results", {"source": "research"})
]

vector_store = VectorStore(embed_dim=768)
vector_store.set_embed_model(embed_model)
vector_store.add_documents(docs)

query = "What is deep learning framework?"
results = vector_store.search(query, top_k=2)

print(f"Query: '{query}'")
print(f"\nTop 2 results:")
for i, r in enumerate(results):
    print(f"  {i+1}. Score: {r['score']:.4f}")
    print(f"     Content: {r['document'].content[:50]}...")

# RAG系统核心组件
## RAG System Core Components

In [ ]:
class RAGSystem:
    """
    完整的RAG系统 / Complete RAG system
    包括：检索 -> 增强 -> 生成
    Including: Retrieve -> Augment -> Generate
    """
    def __init__(self, retriever, generator, embed_model):
        self.retriever = retriever  # 向量数据库 / Vector DB
        self.generator = generator    # LLM / LLM
        self.embed_model = embed_model
    
    def retrieve(self, query, top_k=3):
        """检索相关文档 / Retrieve relevant documents"""
        return self.retriever.search(query, top_k)
    
    def augment(self, query, retrieved_docs):
        """
        将检索结果与查询结合
        Combine retrieved results with query
        
        构建prompt格式：
        Build prompt format:
        [Context]
        {retrieved content}
        
        [Question]
        {query}
        """
        context_parts = []
        context_parts.append("[Context]")
        for i, doc in enumerate(retrieved_docs):
            context_parts.append(f"[{i+1}] {doc['document'].content}")
            if doc['document'].metadata:
                context_parts.append(f"    Source: {doc['document'].metadata.get('source', 'unknown')}")
        
        context_parts.append("")
        context_parts.append("[Question]")
        context_parts.append(query)
        
        augmented_prompt = "\n".join(context_parts)
        return augmented_prompt
    
    def generate(self, prompt):
        """生成回答（模拟LLM）/ Generate answer (simulated LLM)"""
        # 在实际应用中，这里会调用真实的LLM
        # In real application, call actual LLM here
        return f"Based on the retrieved context, here is the answer..."
    
    def answer(self, query, top_k=3):
        """完整的RAG流程 / Complete RAG pipeline"""
        # 1. 检索 / Retrieve
        retrieved = self.retrieve(query, top_k)
        
        # 2. 增强 / Augment
        prompt = self.augment(query, retrieved)
        
        # 3. 生成 / Generate
        answer = self.generate(prompt)
        
        return {
            'answer': answer,
            'retrieved_docs': retrieved,
            'prompt': prompt
        }

# 测试RAG系统 / Test RAG system
class MockLLM:
    """模拟LLM / Mock LLM"""
    def __init__(self, model_name="gpt-4"):
        self.model_name = model_name
    
    def generate(self, prompt):
        return f"[Mock {self.model_name} response]"

llm = MockLLM()
rag = RAGSystem(vector_store, llm, embed_model)

query = "What is PyTorch?"
result = rag.answer(query, top_k=2)

print(f"Query: {query}")
print(f"\nRetrieved {len(result['retrieved_docs'])} documents:")
for doc in result['retrieved_docs']:
    print(f"  - {doc['document'].content[:50]}... (score: {doc['score']:.4f})")
print(f"\nAnswer: {result['answer']}")

# 语义分割与索引优化
## Chunking and Indexing Optimization

In [ ]:
class SemanticChunker:
    """
    语义分块：将文档分割成有意义的小块
    Semantic chunking: split documents into meaningful chunks
    """
    def __init__(self, chunk_size=100, overlap=20):
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def chunk_text(self, text):
        """
        将文本分块（简单滑动窗口）
        Chunk text (simple sliding window)
        """
        words = text.split()
        chunks = []
        start = 0
        
        while start < len(words):
            end = min(start + self.chunk_size, len(words))
            chunk = ' '.join(words[start:end])
            chunks.append(chunk)
            start = end - self.overlap  # 滑动窗口 / Sliding window
        
        return chunks
    
    def chunk_with_overlap(self, text):
        """
        重叠分块以保持上下文连贯性
        Overlapping chunks to maintain context
        """
        words = text.split()
        chunks = []
        start = 0
        
        while start < len(words):
            end = min(start + self.chunk_size, len(words))
            chunk_text = ' '.join(words[start:end])
            
            # 添加元信息 / Add metadata
            chunk = Document(chunk_text, {
                'start_idx': start,
                'end_idx': end,
                'chunk_id': len(chunks)
            })
            chunks.append(chunk)
            
            start = end - self.overlap
            if start >= len(words) - self.overlap:
                break
        
        return chunks

# 测试分块 / Test chunking
chunker = SemanticChunker(chunk_size=20, overlap=5)

long_text = """
PyTorch is an open source machine learning framework developed by Facebook's AI Research lab. 
It provides a wide range of tools for building neural networks, including automatic differentiation, 
GPU acceleration, and a large collection of pre-trained models. PyTorch has become one of the 
most popular deep learning frameworks due to its dynamic computation graph and intuitive API.
"""

chunks = chunker.chunk_with_overlap(long_text)

print(f"Text length: {len(long_text.split())} words")
print(f"Number of chunks: {len(chunks)}")
print(f"\nChunks:")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {chunk.content[:50]}... (idx: {chunk.metadata['start_idx']}-{chunk.metadata['end_idx']})")

# RAG优化技术
## RAG Optimization Techniques

In [ ]:
class HybridRetriever:
    """
    混合检索：结合密集检索和稀疏检索
    Hybrid retrieval: combines dense and sparse retrieval
    """
    def __init__(self, dense_weight=0.7, sparse_weight=0.3):
        self.dense_weight = dense_weight
        self.sparse_weight = sparse_weight
    
    def retrieve(self, query, dense_results, sparse_results):
        """
        合并密集和稀疏检索结果
        Merge dense and sparse retrieval results
        """
        # 简单RRF (Reciprocal Rank Fusion) / Simple RRF
        scores = defaultdict(float)
        
        for rank, result in enumerate(dense_results):
            doc_id = id(result['document'])
            scores[doc_id] += self.dense_weight / (rank + 1)
        
        for rank, result in enumerate(sparse_results):
            doc_id = id(result['document'])
            scores[doc_id] += self.sparse_weight / (rank + 1)
        
        # 排序 / Sort
        sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        
        return [(doc_id, score) for doc_id, score in sorted_docs[:5]]

class Reranker:
    """
    重排序：使用交叉编码器对检索结果重新排序
    Reranking: use cross-encoder to rerank retrieval results
    """
    def __init__(self):
        self.model = None  # 交叉编码器 / Cross-encoder
    
    def rerank(self, query, documents, top_k=3):
        """
        对文档重新排序
        Rerank documents
        """
        # 简化实现：基于关键词匹配评分
        # Simplified: keyword matching based scoring
        query_terms = set(query.lower().split())
        scores = []
        
        for doc in documents:
            doc_terms = set(doc['document'].content.lower().split())
            overlap = len(query_terms & doc_terms)
            score = overlap / max(len(query_terms), 1)
            scores.append((doc, score))
        
        # 排序 / Sort
        scores.sort(key=lambda x: x[1], reverse=True)
        
        return [doc for doc, score in scores[:top_k]]

# 测试重排序 / Test reranking
reranker = Reranker()

query = "PyTorch deep learning"
test_docs = [
    {'document': Document("Python is a programming language", {}), 'score': 0.8},
    {'document': Document("PyTorch is a deep learning framework", {}), 'score': 0.6},
    {'document': Document("Deep learning uses neural networks", {}), 'score': 0.7},
    {'document': Document("Machine learning is AI subset", {}), 'score': 0.5}
]

reranked = reranker.rerank(query, test_docs, top_k=3)

print("Reranked results:")
for i, doc in enumerate(reranked):
    print(f"  {i+1}. {doc['document'].content}")

# RAG评估指标
## RAG Evaluation Metrics

# RAG可视化
## RAG Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# RAG System Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Retrieval precision vs top_k
ax1 = axes[0]
top_k = [1, 3, 5, 10]
precision = [0.85, 0.72, 0.65, 0.52]
ax1.plot(top_k, precision, 'b-o', linewidth=2, markersize=8)
ax1.set_xlabel('Top-K Retrieved')
ax1.set_ylabel('Precision@K')
ax1.set_title('RAG检索精度 / Retrieval Precision')
ax1.grid(True, alpha=0.3)

# 2. Retrieval latency comparison
ax2 = axes[1]
methods = ['Dense', 'Sparse', 'Hybrid']
latency = [45, 32, 58]
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax2.bar(methods, latency, color=colors)
ax2.set_ylabel('Latency (ms)')
ax2.set_title('检索方法延迟对比 / Retrieval Latency')
for bar, lat in zip(bars, latency):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{lat}ms', ha='center', va='bottom')

# 3. RAG Pipeline flow
ax3 = axes[2]
ax3.axis('off')
pipeline_steps = ['Query', 'Embedding', 'Vector Search', 'Retrieve Docs', 'Augment Prompt', 'LLM Generate']
y_pos = np.arange(len(pipeline_steps))
ax3.barh(y_pos, [1]*len(pipeline_steps), color=plt.cm.Blues(np.linspace(0.3, 0.9, len(pipeline_steps))))
ax3.set_yticks(y_pos)
ax3.set_yticklabels(pipeline_steps)
ax3.set_xlabel('Pipeline Stage')
ax3.set_title('RAG流程 / RAG Pipeline')

plt.tight_layout()
plt.savefig('../images/rag_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("RAG Visualization saved!")

In [ ]:
def evaluate_rag(rag_system, test_queries, ground_truth):
    """
    评估RAG系统
    Evaluate RAG system
    
    指标：
    - Precision@K: 检索到的相关文档比例
    - MRR: 平均倒数排名
    - RAGAS: 答案质量评估
    """
    metrics = {
        'precision_at_1': [],
        'precision_at_3': [],
        'mrr': []
    }
    
    for query, relevant_docs in zip(test_queries, ground_truth):
        results = rag_system.retrieve(query, top_k=3)
        
        # Precision@K
        retrieved_docs = [r['document'].content for r in results]
        relevant_set = set(relevant_docs)
        
        # P@1
        p1 = 1 if retrieved_docs[0] in relevant_set else 0
        metrics['precision_at_1'].append(p1)
        
        # P@3
        p3 = len(set(retrieved_docs) & relevant_set) / min(3, len(relevant_docs))
        metrics['precision_at_3'].append(p3)
        
        # MRR
        for i, doc in enumerate(retrieved_docs):
            if doc in relevant_set:
                metrics['mrr'].append(1 / (i + 1))
                break
        else:
            metrics['mrr'].append(0)
    
    return {k: np.mean(v) for k, v in metrics.items()}

# 模拟评估 / Simulated evaluation
test_queries = [
    "What is PyTorch?",
    "What is deep learning?",
    "What is machine learning?"
]

ground_truth = [
    ["PyTorch is a deep learning framework"],
    ["Deep learning uses neural networks"],
    ["Machine learning is AI subset"]
]

eval_results = evaluate_rag(rag, test_queries, ground_truth)

print("RAG Evaluation Results:")
for metric, value in eval_results.items():
    print(f"  {metric}: {value:.4f}")

# 总结

| 组件 | 功能 | 实现要点 |
|------|------|----------|
| Embedding Model | 将文本转为向量 | 词嵌入平均（简化实现） |
| Vector Store | 存储和检索向量 | 余弦相似度检索 |
| Chunking | 文档分割 | 滑动窗口 + overlap |
| Retriever | 找到相关文档 | 密集 + 稀疏混合 |
| Reranker | 重排序结果 | 关键词匹配 |
| Generator | 生成答案 | LLM (GPT-4/Claude) |

RAG是当前LLM应用的核心架构，可以显著提升模型的知识时效性和准确性。

In [ ]:
# RAG检索流程详解 / RAG Retrieval Process Detail
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Vector similarity search / 向量相似度搜索
ax1 = axes[0]
queries = ['What is PyTorch?', 'Deep learning framework', 'Neural network library']
docs = ['Python language', 'PyTorch framework', 'Machine learning', 'Deep learning', 'Facebook AI research']
similarities = [
    [0.3, 0.85, 0.5, 0.7, 0.6],
    [0.4, 0.8, 0.3, 0.75, 0.65],
    [0.2, 0.7, 0.4, 0.9, 0.5]
]

im = ax1.imshow(similarities, cmap='Blues', aspect='auto')
ax1.set_xticks(range(len(docs)))
ax1.set_yticks(range(len(queries)))
ax1.set_xticklabels(docs, rotation=45, ha='right')
ax1.set_yticklabels(queries)
ax1.set_title('Query-Document Similarity Matrix
(Higher = More Relevant)')
plt.colorbar(im, ax=ax1)

# 2. RAG retrieval steps / RAG检索步骤
ax2 = axes[1]
ax2.axis('off')

steps = ['1.Query', '2.Embed', '3.Search', '4.Retrieverank', '5.Combine', '6.Generate']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']

for i, (step, color) in enumerate(zip(steps, colors)):
    circle = plt.Circle((0.2 + i*1.5, 0.5), 0.35, facecolor=color, edgecolor='black', linewidth=2)
    ax2.add_patch(circle)
    ax2.text(0.2 + i*1.5, 0.5, step.replace('.', '
'), ha='center', va='center', fontsize=7, fontweight='bold')

ax2.set_xlim(-0.2, len(steps)*1.5)
ax2.set_ylim(0, 1)
ax2.set_title('RAG Retrieval Pipeline
(Step by Step)')

# 3. Context window impact / 上下文窗口影响
ax3 = axes[2]
window_sizes = [512, 1024, 2048, 4096, 8192, 16384]
retrieval_scores = [0.95, 0.92, 0.88, 0.82, 0.75, 0.65]
context_fidelity = [0.98, 0.95, 0.90, 0.82, 0.70, 0.55]

ax3.plot(window_sizes, retrieval_scores, 'b-o', linewidth=2, markersize=8, label='Retrieval Score')
ax3.plot(window_sizes, context_fidelity, 'r-s', linewidth=2, markersize=8, label='Context Fidelity')
ax3.set_xlabel('Context Window Size (tokens)')
ax3.set_ylabel('Score')
ax3.set_title('Context Window Impact on RAG
(Larger window = more context but dilute relevance)')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../images/rag_retrieval_process.png', dpi=150, bbox_inches='tight')
plt.show()

print("RAG retrieval process visualization saved!")

# 已实现 / Implemented

本notebook已完整实现以下内容：

1. **向量数据库** - 简化版VectorStore
2. **RAG系统** - 检索→增强→生成流程
3. **语义分块** - 重叠滑动窗口分块
4. **混合检索** - Dense + Sparse (RRF融合)
5. **重排序** - 基于关键词匹配的Reranker
6. **RAG评估指标** - Precision@K, MRR

## 扩展阅读 / Further Reading

| 主题 | 说明 | 推荐资源 |
|------|------|----------|
| **Sentence-BERT** | 更好的语义嵌入 | [SBERT](https://www.sbert.net/) |
| **HyDE** | 假设性文档嵌入 | [HyDE Paper](https://arxiv.org/abs/2212.10496) |
| **Self-RAG** | 自适应检索增强 | [Self-RAG](https://arxiv.org/abs/2312.05917) |
| **RAG评估** | RAGAS评估指标 | [RAGAS Paper](https://arxiv.org/abs/2309.15217) |
| **向量数据库** | Milvus/Pinecone | [Milvus](https://milvus.io/) |
